# Advanced I/O: BytesIO Priority & PDF Processing
**Target Object:** `BytesIO` as the primary interface for in-memory PDF handling.

This notebook demonstrates the transition from basic file paths to memory-efficient binary streams, concluding with automated slug generation logic.

In [24]:
import re
import requests
from io import BytesIO, StringIO
from PyPDF2 import PdfReader, PdfWriter

## 1. Documentation: Why BytesIO?

| Feature | Description |
| :--- | :--- |
| **Memory Efficiency** | Processes PDFs without writing to the physical disk. |
| **Speed** | 10x-50x faster than disk I/O for small to medium files. |
| **Integration** | Direct compatibility with `requests.content` and web frameworks (Django/FastAPI). |
| **Constraint** | Must use `.seek(0)` after writing to a buffer before reading it back. |

## 2. Primary Use Case: BytesIO (The Priority)
Handling a PDF downloaded from a URL entirely in memory.

In [ ]:
def process_pdf_from_url(url):
    # 1. Fetch binary data
    response = requests.get(url)
    
    # 2. Wrap in BytesIO (Virtual File)
    pdf_buffer = BytesIO(response.content)
    
    # 3. Read directly from buffer
    reader = PdfReader(pdf_buffer)
    
    print(f"Source: {url}")
    print(f"Page Count: {len(reader.pages)}")
    return reader

# Example (Conceptual)
process_pdf_from_url("https://www.w3.org/WAI/ER/tests/xhtml/testfiles/resources/pdf/dummy.pdf")

Source: https://www.w3.org/WAI/ER/tests/xhtml/testfiles/resources/pdf/dummy.pdf
Page Count: 1


## 3. Secondary Use Case: StringIO
Used specifically for **text extraction buffers**. Note that `PdfReader` cannot read from `StringIO` because PDFs are binary, not text.

In [ ]:
def extract_text_to_string_buffer(reader):
    text_buffer = StringIO()
    
    for page in reader.pages:
        text_buffer.write(page.extract_text())
    
    content = text_buffer.getvalue()
    text_buffer.close()
    return content

## 4. Advanced: Auto-Slug Generation
Generating a URL-friendly slug from the PDF's internal metadata 'Name' attribute.

In [33]:
def generate_slug_from_reader(reader):
    # Retrieve metadata title
    name = reader.metadata.title or "Untitled Document"
    
    # Slugification Logic
    slug = name.lower().strip()
    slug = re.sub(r'[^a-z0-9\s-]', '', slug) # Remove special chars
    slug = re.sub(r'[\s-]+', '-', slug)      # Replace spaces with dashes
    
    return slug

# Execution Flow:
# 1. BytesIO -> PdfReader
# 2. PdfReader -> Slug
with open('sample.pdf', 'rb') as f:
    buffer = BytesIO(f.read())
    reader = PdfReader(buffer)
    print(f"Final Slug: {generate_slug_from_reader(reader)}")

Final Slug: anonymous


In your code, the **vector store is not a separate “database system”**—it’s a **Python dictionary that you manually build in memory** to act like a vector database.

Let’s break down exactly what your `build_vector_store()` function is doing in simple terms.

---

# 🧠 What your vector store function actually does

This function:

```python
build_vector_store(uploaded_files, chunk_size=400, overlap=100)
```

👉 Converts PDFs into a **searchable AI memory system**

---

# ⚙️ Step-by-step what happens

## 1. 📥 Reads PDF files

```python
text = read_pdf(uploaded_file.read())
```

* Takes raw PDF bytes
* Extracts all text from it

✔ Output: full document text

---

## 2. ✂️ Splits text into chunks

```python
file_chunks = split_text(text, chunk_size, overlap)
```

* Breaks big text into smaller parts (400 words each)
* Adds overlap (100 words) so meaning is not lost

✔ Output:

```text
["chunk 1", "chunk 2", "chunk 3", ...]
```

---

## 3. 🏷️ Stores chunk + source info

```python
chunks.append(chunk)
sources.append(f"{file_name} chunk {index}")
```

So each piece knows:

* which file it came from
* which chunk number it is

✔ Example:

```text
"book.pdf chunk 3"
```

---

## 4. 🧠 Converts chunks into vectors (MOST IMPORTANT)

```python
embeddings = embed_texts(chunks)
```

This is the **AI brain step**:

Each chunk becomes a vector like:

```text
"AI is learning from data"
→ [0.12, -0.44, 0.91, ...]
```

So meaning is stored mathematically.

---

## 5. 📦 Builds the “vector store”

```python
return {
    "chunks": chunks,
    "embeddings": embeddings,
    "sources": sources
}
```

This is your vector store.

---

# 🧠 So what is your vector store REALLY?

Your vector store is:

```text
A Python dictionary containing:
```

| Key        | Meaning                    |
| ---------- | -------------------------- |
| chunks     | original text pieces       |
| embeddings | meaning vectors            |
| sources    | where each chunk came from |

---

# 🔍 How it is used later

When user asks a question:

### 1. Question becomes vector

```python
question_embedding = embed_texts([question])[0]
```

---

### 2. Compare with all chunks

```python
scores = store["embeddings"] @ question_embedding
```

👉 This finds similarity between question and all chunks

---

### 3. Pick best matches

```python
top_indexes = np.argsort(-scores)[:top_k]
```

---

### 4. Return relevant text

```python
store["chunks"][index]
```

---

# 🧭 Simple mental model

Your system works like this:

```
📄 PDFs
   ↓
✂️ split into chunks
   ↓
🧠 convert chunks → vectors
   ↓
📦 store everything in dictionary
   ↓
❓ user question
   ↓
🧠 convert question → vector
   ↓
🔍 find closest chunks
   ↓
📄 return best answers
```

---

# ⚠️ Important truth

Your “vector store” is:

❌ NOT a real database
❌ NOT FAISS / Pinecone / Weaviate

It is:

✔ an **in-memory vector index using NumPy arrays**

---

# 🚀 If you want to upgrade it

You can turn this into a real system using:

* FAISS → fast local search
* Chroma → easy persistent storage
* Pinecone → production scale

---

# 💡 One-line explanation of YOUR function

👉 Your `build_vector_store()` function takes PDFs, splits them into chunks, converts them into embeddings, and stores everything in a Python dictionary so you can later do semantic search on them.

---

